# Семинар 08. Исключения и контекстные менеджеры

Пять задач про то, что делать, когда операция не может выполнить обещанное: подставить запасное
значение, назвать причину отказа, продолжить после плохой строки и откатить пачку изменений
целиком.

## Цели

- поднимать `ValueError` с понятным кодом причины и соблюдать порядок проверок;
- перехватывать только ожидаемую ошибку и подставлять запасное значение;
- понимать, когда выполняются `else` и `finally`, и опираться на этот порядок;
- продолжать обработку после неудачной записи, сохраняя её номер;
- возвращать данные в прежнее состояние, если пачка операций сорвалась.

## Перед началом

Откройте [лекцию](lecture.ipynb). Задания идут от простых к сложным и разбираются подряд;
задание 6 — дополнительное, для тех, кто закончил раньше. Классы, декораторы и собственные
контекстные менеджеры не нужны.

Всё используемое входит в стандартную библиотеку и работает на Python 3.10 и новее, поэтому
ноутбук открывается и локально, и в Colab. Внешние файлы не понадобятся: поток в последнем
задании создаётся в памяти.

Замените `raise NotImplementedError` своим кодом. Сначала запустите ячейку с определением
функции, затем ячейку проверок под ней. До появления решения проверки намеренно не проходят.
**Ячейку с `error_code` выполните один раз** — она нужна проверкам нескольких заданий.

Внутри функций не используйте `input` и `print`. Входные данные не меняйте — кроме задания 5,
где изменение словаря и есть предмет задачи, и журнала в задании 3.

## Разминка. Что напечатает

Четыре фрагмента про порядок выполнения. Сначала запишите прогноз, затем выполните ячейку и
сверьтесь. Разбор — под спойлером после всех фрагментов.

In [ ]:
# Фрагмент 1
def run(text):
    journal = []
    try:
        journal.append("try")
        int(text)
    except ValueError:
        journal.append("except")
    else:
        journal.append("else")
    finally:
        journal.append("finally")
    return journal


print("int('7'):  ", run("7"))
print("int('abc'):", run("abc"))

In [ ]:
# Фрагмент 2
try:
    int("abc")
except Exception:
    print("сработал except Exception")
except ValueError:
    print("сработал except ValueError")

In [ ]:
# Фрагмент 3
def answer():
    try:
        return "из try"
    finally:
        return "из finally"


print(answer())

In [ ]:
# Фрагмент 4
balances = {"knight": 100, "witch": 50}

try:
    balances["knight"] -= 30
    balances["pekka"] -= 10
except KeyError as error:
    print("KeyError:", error)

print(balances)

<details>
<summary>Разбор</summary>

**Фрагмент 1.** `['try', 'else', 'finally']` и `['try', 'except', 'finally']`. `else`
выполняется только тогда, когда в `try` ничего не случилось, `except` — только когда ошибка
была, а `finally` — всегда и последним. Отсюда простое правило: действие «если всё получилось»
кладут в `else`, а действие «в любом случае» — в `finally`.

**Фрагмент 2.** Печатается `сработал except Exception`. Обработчики проверяются сверху вниз,
и первый подходящий забирает исключение. `ValueError` — потомок `Exception`, поэтому до второго
обработчика дело не доходит: он недостижим. Конкретные типы пишут выше общих.

**Фрагмент 3.** `из finally`. Возврат из `finally` заменяет уже подготовленный возврат из
`try` — так же он заменил бы и исключение. Поэтому `return` и `raise` внутри `finally`
прячут то, ради чего блок выполнялся.

**Фрагмент 4.** `KeyError: 'pekka'`, затем `{'knight': 70, 'witch': 50}`. Исключение прервало
работу, но первое изменение уже произошло и само не откатилось. Если операция обещает «всё или
ничего», возвращать прежние значения придётся вручную.

</details>

## Как проверки смотрят на ошибку

Проверка должна упасть и тогда, когда исключения не было вовсе. Поэтому в ячейке ниже
определена небольшая функция: она возвращает текст `ValueError`, если он случился, и `None`,
если вызов прошёл спокойно. Это не решение какого-либо задания — выполните ячейку один раз.

In [ ]:
def error_code(call):
    try:
        call()
    except ValueError as error:
        return str(error)
    return None


print(error_code(lambda: int("abc")))
print(error_code(lambda: int("7")))

### Задание 1. Списание с кодом причины

Напишите `spend(balance, amount)`: верните остаток после списания.

Если списание невозможно, поднимите `ValueError`, текст которого — **код причины**:

- `amount` — сумма не больше нуля;
- `funds` — на счёте меньше денег, чем просят.

Проверки идут в этом порядке: при неположительной сумме причина всегда `amount`, даже если
денег и так не хватает. Списать весь остаток можно. `balance` неотрицателен, оба числа целые,
по модулю не больше `10**9`.

Примеры: `spend(100, 30) == 70`, `spend(100, 100) == 0`; у `spend(100, 0)` код `amount`, у
`spend(100, 101)` — `funds`.

In [ ]:
def spend(balance: int, amount: int) -> int:
    raise NotImplementedError

In [ ]:
assert spend(100, 30) == 70
assert spend(100, 100) == 0
assert spend(10**9, 10**9) == 0
assert error_code(lambda: spend(100, 30)) is None

assert error_code(lambda: spend(100, 0)) == "amount"
assert error_code(lambda: spend(100, -5)) == "amount"
assert error_code(lambda: spend(100, 101)) == "funds"
assert error_code(lambda: spend(0, 1)) == "funds"

# Порядок проверок: неположительная сумма важнее нехватки денег.
assert error_code(lambda: spend(0, -1)) == "amount"

### Задание 2. Запасное значение

Напишите `parse_amounts(texts, default)`: верните пару «список чисел, сколько раз подставлено
запасное значение».

Для каждой строки попробуйте получить целое число правилами `int()`. Если не получилось,
поставьте в ответ `default` и учтите замену. Функция не должна падать ни на какой строке.

`int()` сам допускает пробелы по краям, знак, ведущие нули и подчёркивания между цифрами:
`" 7 "`, `"+5"`, `"007"`, `"1_000"` — числа. А `"abc"`, `"3.5"`, `"1e3"`, `"0x10"` и пустая
строка — нет.

До 100 000 строк длиной до 20 символов, `default` — целое число. Входной список не меняйте.

Пример: `parse_amounts(["42", "abc", " 7 ", "3.5"], -1)` вернёт `([42, -1, 7, -1], 2)`.

In [ ]:
def parse_amounts(texts: list[str], default: int) -> tuple[list[int], int]:
    raise NotImplementedError

In [ ]:
assert parse_amounts([], 0) == ([], 0)
assert parse_amounts(["42", "abc", " 7 ", "3.5"], -1) == ([42, -1, 7, -1], 2)
assert parse_amounts(["+5", "-3", "007", "1_000"], 0) == ([5, -3, 7, 1000], 0)
assert parse_amounts(["", "   "], 9) == ([9, 9], 2)
assert parse_amounts(["1e3", "0x10"], 0) == ([0, 0], 2)
assert parse_amounts(["abc"], 0) == ([0], 1)

texts = ["5", "нет"]
before = texts.copy()
assert parse_amounts(texts, 1) == ([5, 1], 1)
assert texts == before

### Задание 3. Журнал разбора

Напишите `record_parse(text, journal)`: разберите строку правилами `int()` и запишите ход
работы в список `journal`.

Порядок записей:

1. `"start"` — до попытки разбора;
2. `"ok"` — только если разбор удался;
3. `"error"` — только если `int()` не справился;
4. `"finish"` — последней записью, в любом случае.

Верните число при успехе и `None` при неудаче: исключение наружу не выходит. Журнал меняется на
месте, прежние записи сохраняются. Ноль — успешный разбор, а не ошибка.

Запись `"finish"` должна стоять последней и тогда, когда функция уже решила, что вернёт.

Примеры: после `record_parse("7", journal)` в журнале `["start", "ok", "finish"]`, после
`record_parse("abc", journal)` — `["start", "error", "finish"]`.

In [ ]:
def record_parse(text: str, journal: list[str]) -> int | None:
    raise NotImplementedError

In [ ]:
journal: list[str] = []
assert record_parse("7", journal) == 7
assert journal == ["start", "ok", "finish"]

assert record_parse("abc", journal) is None
assert journal == ["start", "ok", "finish", "start", "error", "finish"]

assert record_parse("0", []) == 0

previous = ["прежняя запись"]
assert record_parse(" 42 ", previous) == 42
assert previous == ["прежняя запись", "start", "ok", "finish"]

empty_journal: list[str] = []
assert record_parse("", empty_journal) is None
assert empty_journal == ["start", "error", "finish"]

### Задание 4. Пакет заявок

Напишите `process_requests(initial, requests)`: каждая строка списка — сумма одного списания.
Выполните допустимые списания по порядку и верните пару «остаток, номера отклонённых заявок».

Заявка отклоняется, если сумма не разбирается правилами `int()`, не положительна или на счёте
недостаточно денег. Отклонённая заявка не меняет остаток и не мешает обработать следующие.
Номера начинаются с единицы, пустая строка — тоже заявка.

Начальный остаток неотрицателен, до 100 000 строк длиной до 100 символов, суммы по модулю не
больше `10**9`. Входной список не меняйте.

Пример: при остатке 100 заявки `["30", "80", "-2", "ошибка", "70"]` дают `(0, [2, 3, 4])`:
восьмидесяти уже нет, минус два недопустимы, «ошибка» не разбирается, а последние 70 списались
ровно под ноль.

In [ ]:
def process_requests(initial: int, requests: list[str]) -> tuple[int, list[int]]:
    raise NotImplementedError

In [ ]:
assert process_requests(100, []) == (100, [])
assert process_requests(100, ["30", "80", "-2", "ошибка", "70"]) == (0, [2, 3, 4])
assert process_requests(100, ["100"]) == (0, [])
assert process_requests(100, ["101"]) == (100, [1])
assert process_requests(100, ["0", ""]) == (100, [1, 2])
assert process_requests(0, ["1"]) == (0, [1])
assert process_requests(100, [" 40 ", "007"]) == (53, [])

# Отклонённая заявка не мешает следующим.
assert process_requests(50, ["нет", "20", "999", "30"]) == (0, [1, 3])

requests = ["10", "нет"]
before = requests.copy()
assert process_requests(100, requests) == (90, [2])
assert requests == before

### Задание 5. Пачка операций целиком

Счета хранятся в словаре «название → остаток». Пачка — список операций
`("deposit", имя, сумма)` и `("withdraw", имя, сумма)`.

Напишите `apply_batch(balances, operations)`: выполняйте операции по порядку. Если очередная
операция недопустима, отмените изменения, уже сделанные предыдущими операциями этой пачки, и
остановитесь.

Операция недопустима, если сумма не больше нуля, такого счёта нет или на счёте не хватает
денег. Верните номер первой недопустимой операции, считая с единицы, или `0`, если выполнились
все.

Словарь `balances` меняется на месте — это единственное задание, где вход изменяется. При
неудаче он должен остаться ровно таким, каким был до пачки. Счетов и операций до 100 000,
поэтому копировать весь словарь перед пачкой нельзя: запоминайте прежние значения только тех
счетов, которых коснулись. Отмену напишите в одном месте, а не повторяйте после каждой
проверки.

Пример: при `{"knight": 100, "witch": 50}` пачка
`[("deposit", "knight", 20), ("withdraw", "witch", 70)]` вернёт `2`, а словарь останется
прежним.

In [ ]:
def apply_batch(balances: dict[str, int], operations: list[tuple]) -> int:
    raise NotImplementedError

In [ ]:
balances = {"knight": 100, "witch": 50}
assert apply_batch(balances, []) == 0
assert balances == {"knight": 100, "witch": 50}

assert apply_batch(balances, [("deposit", "knight", 20), ("withdraw", "witch", 70)]) == 2
assert balances == {"knight": 100, "witch": 50}

assert apply_batch(balances, [("withdraw", "knight", 30), ("deposit", "witch", 5)]) == 0
assert balances == {"knight": 70, "witch": 55}

assert apply_batch(balances, [("deposit", "pekka", 1)]) == 1
assert apply_batch(balances, [("withdraw", "knight", 0)]) == 1
assert apply_batch(balances, [("withdraw", "knight", 71)]) == 1
assert balances == {"knight": 70, "witch": 55}

# Списать ровно весь остаток можно.
assert apply_batch(balances, [("withdraw", "knight", 70)]) == 0
assert balances == {"knight": 0, "witch": 55}

# Счёт, изменённый в пачке дважды, возвращается к самому первому значению.
twice = {"a": 10}
assert apply_batch(twice, [
    ("deposit", "a", 5), ("deposit", "a", 5), ("withdraw", "a", 100),
]) == 3
assert twice == {"a": 10}

## Дополнительно. Среднее из открытого потока

Задание для тех, кто закончил раньше.

Напишите `read_mean(source)`: функция получает **уже открытый** текстовый поток — файл или
`StringIO`. Прочитайте целые неотрицательные числа, по одному в строке; строки, пустые после
удаления пробелов, пропускаются. Верните среднее арифметическое как `float`.

- Нечисловая или отрицательная значимая строка — `ValueError`.
- Если значимых строк не нашлось, тоже `ValueError`.
- Функция принимает поток во владение: **после любого выхода он закрыт**, в том числе при ошибке.
- Чтение и подсчёт должны происходить, пока поток ещё открыт.

До 100 000 строк длиной до 100 символов. Внешние файлы не нужны: в проверках поток создаётся в
памяти через `StringIO` — он поддерживает чтение, `with` и свойство `.closed` так же, как
файл.

In [ ]:
from io import StringIO


def read_mean(source) -> float:
    raise NotImplementedError

In [ ]:
stream = StringIO("1\n2\n3\n")
assert read_mean(stream) == 2.0
assert stream.closed

spaces = StringIO("  \n4\n\n 6 \n")
assert read_mean(spaces) == 5.0
assert spaces.closed

broken = StringIO("1\nabc\n")
assert error_code(lambda: read_mean(broken)) is not None
assert broken.closed

negative = StringIO("5\n-1\n")
assert error_code(lambda: read_mean(negative)) is not None
assert negative.closed

nothing = StringIO("   \n\n")
assert error_code(lambda: read_mean(nothing)) is not None
assert nothing.closed

single = StringIO("7")
assert read_mean(single) == 7.0
assert single.closed

## Итоги

- Код причины в тексте исключения отвечает на вопрос «что именно не так», а порядок проверок
  решает, какая причина победит, когда нарушено сразу несколько правил.
- `except` ставят вокруг той операции, которая может не получиться, и ловят конкретный тип:
  широкий `except` прячет чужие ошибки.
- `else` выполняется только при успехе, `finally` — всегда и последним; `return` и `raise`
  внутри `finally` заменяют собой то, ради чего блок выполнялся.
- Исключение не отменяет изменения, которые уже произошли. Если операция обещает «всё или
  ничего», прежние значения надо запоминать и возвращать вручную.
- `with` закрывает поток и при обычном выходе, и при ошибке.

[Домашка](https://contest.yandex.ru/contest/CONTEST_ID/enter/?retPage=).